# Phase 3: Dice-SGD Hyperparameter Grid Search (CIFAR-10)
**Project:** Mitigating Clipping Bias in DP-SGD via Error Feedback (Dice-SGD)  
**Authors:** Priyanshu Agarwal & Sudiksha Singh

### Objective
Standard DP-SGD is highly fragile regarding hyperparameter selection. This notebook systematically tests the robustness of our custom `DiceSGDOptimizer` across a 3D grid of configurations on the CIFAR-10 dataset using a refactored ResNet-18 architecture.

**Engineering Roadblocks Addressed:**
1. **Opacus Architecture Compliance:** Standard ResNet-18 uses `BatchNorm`, which violates DP assumptions. We use Opacus's `ModuleValidator` to automatically refactor the network to use `GroupNorm`.
2. **VRAM Constraints:** To accommodate the 3x memory footprint of the error-feedback buffer, batch size is strictly capped at `64`.
3. **Multiprocessing Deadlocks:** Asynchronous data loading (`num_workers>0`) causes GPU deadlocks when repeatedly initializing the privacy engine across 36 runs. We enforce `num_workers=0`.

In [ ]:
# Install dependencies
!pip install -q torch torchvision opacus pandas

### 1. The Custom Optimizer (Dice-SGD)
We redefine our custom error-feedback optimizer here so the notebook remains fully self-contained.

In [ ]:
import torch
from torch.optim import SGD
from opacus.optimizers import DPOptimizer

class DiceSGDOptimizer(DPOptimizer):
    """Custom Opacus Optimizer implementing Error Feedback (Dice-SGD)."""
    def __init__(self, optimizer: SGD, *, noise_multiplier: float, max_grad_norm: float, expected_batch_size: int, **kwargs):
        super().__init__(
            optimizer=optimizer, noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm, expected_batch_size=expected_batch_size, **kwargs
        )
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad:
                    self.state[p]['error_buffer'] = torch.zeros_like(p.data)

    def step(self, closure=None):
        # Inject error directly into per-sample gradients BEFORE Opacus clips them
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad and hasattr(p, 'grad_sample'):
                    e_t = self.state[p]['error_buffer']
                    self.state[p]['unclipped_grad_mean'] = p.grad_sample.mean(dim=0).clone()
                    p.grad_sample.add_(e_t.unsqueeze(0))

        # Native Opacus Clipping and Noising (Maintains epsilon budget)
        super().step(closure)

        # Calculate exactly what was discarded and update the error buffer
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.requires_grad and hasattr(p, 'grad'):
                    unclipped_grad_mean = self.state[p]['unclipped_grad_mean']
                    v_t = p.grad
                    self.state[p]['error_buffer'].add_(unclipped_grad_mean - v_t)

### 2. Environment, Data, and Architecture Setup
Loading CIFAR-10 and fetching the ResNet-18 model. Notice the crucial use of `ModuleValidator.fix()` to make the model privacy-compliant.

In [ ]:
import warnings
import pandas as pd
from torchvision import datasets, transforms, models
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torch import nn
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CIFAR-10 Data Loading
# CRITICAL: num_workers=0 prevents deadlocks during the 36-loop grid search
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

def get_compliant_resnet18():
    """Fetches ResNet18, replaces BatchNorm with GroupNorm, and adjusts for CIFAR-10."""
    model = models.resnet18(num_classes=10)
    # Opacus utility to automatically replace BatchNorm with GroupNorm
    model = ModuleValidator.fix(model)
    return model.to(device)

### 3. Automated 3D Grid Search
We iterate over 36 configurations: $C \in \{0.1, 0.5, 1.0, 5.0\}$, $\sigma \in \{0.5, 1.0, 2.0\}$, and $\eta \in \{0.01, 0.05, 0.1\}$.
Each config runs for 5 epochs. Results are saved iteratively to a CSV for robust checkpointing.

In [ ]:
# Define Grid Search Space (36 combinations)
clip_norms = [0.1, 0.5, 1.0, 5.0]
noise_multipliers = [0.5, 1.0, 2.0]
learning_rates = [0.01, 0.05, 0.1]
epochs_per_run = 5

results = []

print("=== STARTING 3D GRID SEARCH ===")
for C in clip_norms:
    for sigma in noise_multipliers:
        for lr in learning_rates:
            print(f"\nTesting Config -> C: {C} | Sigma: {sigma} | LR: {lr}")

            # 1. Initialize fresh model and engine to prevent state-leaking
            model = get_compliant_resnet18()
            base_optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.5)
            criterion = nn.CrossEntropyLoss()
            privacy_engine = PrivacyEngine()

            model, optimizer, train_loader = privacy_engine.make_private(
                module=model,
                optimizer=base_optimizer,
                data_loader=train_loader,
                noise_multiplier=sigma,
                max_grad_norm=C,
            )

            # Inject Dice-SGD
            dice_optimizer = DiceSGDOptimizer(
                optimizer=optimizer.optimizer,
                noise_multiplier=optimizer.noise_multiplier,
                max_grad_norm=optimizer.max_grad_norm,
                expected_batch_size=optimizer.expected_batch_size
            )
            dice_optimizer.state = optimizer.state

            # 2. Training Loop
            for epoch in range(epochs_per_run):
                model.train()
                epoch_loss = 0
                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    dice_optimizer.zero_grad()
                    output = model(images)
                    loss = criterion(output, labels)
                    loss.backward()
                    dice_optimizer.step()
                    epoch_loss += loss.item()

                avg_loss = epoch_loss / len(train_loader)
                epsilon = privacy_engine.get_epsilon(delta=1e-5)

                print(f"  Epoch {epoch+1} | Loss: {avg_loss:.4f}")

                # Save Data
                results.append({
                    'clip_norm': C,
                    'noise_multiplier': sigma,
                    'learning_rate': lr,
                    'epoch': epoch + 1,
                    'loss': avg_loss,
                    'epsilon': epsilon
                })

            # Free up GPU memory between configurations
            del model, base_optimizer, dice_optimizer, privacy_engine
            torch.cuda.empty_cache()

# Save final results
df = pd.DataFrame(results)
df.to_csv('grid_search_results.csv', index=False)
print("\n=== GRID SEARCH COMPLETE. Saved to 'grid_search_results.csv' ===")

### 4. Analyze Results
Here we load the CSV and display the top 5 configurations to verify our "Optimal Regime" ($C=1.0, \sigma=0.5, \eta=0.01$).

In [ ]:
# Analyze the final epoch of each configuration
final_epochs = df[df['epoch'] == 5]

# Sort by lowest loss to find the best configurations
top_5 = final_epochs.sort_values(by='loss', ascending=True).head(5)

print("--- Top 5 Dice-SGD Configurations ---")
print(top_5[['clip_norm', 'noise_multiplier', 'learning_rate', 'loss', 'epsilon']].to_string(index=False))